# Polynomial Regression für Spirob Winkelerkennung

Dieses Notebook implementiert multivariate polynomiale Regression als Alternative zum LSTM-Ansatz.

## Vorteile:
- Deutlich geringerer Rechenaufwand (nur Matrixmultiplikation)
- Kein Training zur Laufzeit nötig
- Deterministisch und interpretierbar
- Schnellere Inferenz
- Geringerer Speicherbedarf

## Nachteile:
- Weniger flexibel bei komplexen nichtlinearen Beziehungen
- Feature Engineering notwendig
- Overfitting-Gefahr bei zu hohen Polynomgraden

## 1. Imports und Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time
import joblib

# Für schönere Plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("Imports erfolgreich!")

## 2. Daten laden und vorbereiten

Wir verwenden die gleichen Daten wie beim LSTM-Training.

In [ ]:
# Liste der Dateien (gleich wie im nettraining.ipynb)
FILE_PATHS = [
    "runs/2025-12-10_17-12-09/merged.csv",
    "runs/2025-12-12_11-19-07/merged.csv",
    "runs/2025-12-12_14-04-33/merged.csv",
    "runs/2025-12-15_16-18-54/merged.csv",
    "runs/2026-01-19_15-35-32/merged.csv",
]

def process_single_run(csv_path):
    """Liest eine einzelne Datei, pivotisiert und bereinigt sie."""
    print(f"Verarbeite: {csv_path} ...")
    
    # 1. Laden & Sortieren
    df_raw = pd.read_csv(csv_path).sort_values("t_ns")
    
    # 2. Sensoren trennen
    sensors_data = []
    sensor_names = []
    min_len = 1000000000

    for b in range(3):
        for s in range(5):
            # Pandas Filterung
            subset = df_raw[(df_raw["board_id"] == b) & (df_raw["sensor_id"] == s)].copy()
            subset = subset.sort_values("t_ns")
            sensors_data.append(subset)
            sensor_names.append(f"b{b}_s{s}")
            if len(subset) < min_len:
                min_len = len(subset)
    
    # 3. Features bauen (Reshape)
    df_clean_list = []
    
    # Winkel (Labels) vom ersten Sensor nehmen und kürzen
    thetas = sensors_data[0].iloc[:min_len][["theta_1", "theta_2", "theta_3", "theta_4"]].reset_index(drop=True)
    df_clean_list.append(thetas)
    
    # Sensordaten für alle Sensoren
    for i, subset in enumerate(sensors_data):
        df_s = subset.iloc[:min_len].reset_index(drop=True)
        
        # Magnitude berechnen (Feature Engineering)
        mag = np.sqrt(df_s["x"]**2 + df_s["y"]**2 + df_s["z"]**2)
        
        sid = sensor_names[i]
        df_feat = pd.DataFrame({
            f"x_{sid}": df_s["x"],
            f"y_{sid}": df_s["y"],
            f"z_{sid}": df_s["z"],
            f"mag_{sid}": mag
        })
        df_clean_list.append(df_feat)
        
    # Zusammenfügen für diesen Run
    df_run = pd.concat(df_clean_list, axis=1)
    
    # Cleaning (NaNs raus)
    df_run = df_run.replace([np.inf, -np.inf], np.nan).dropna()
    
    return df_run

# Alle Runs verarbeiten
all_runs_data = []

for path in FILE_PATHS:
    try:
        processed_run = process_single_run(path)
        all_runs_data.append(processed_run)
        print(f" -> OK: {len(processed_run)} Samples geladen.")
    except Exception as e:
        print(f" -> FEHLER bei {path}: {e}")

# Alles untereinander kleben (Concatenate)
if all_runs_data:
    df_total = pd.concat(all_runs_data, axis=0, ignore_index=True)
    
    print("-" * 30)
    print(f"GESAMT DATASET: {len(df_total)} Zeilen (Samples)")
    print(f"Features (Spalten): {len(df_total.columns)}")
else:
    print("Keine Daten geladen!")

## 3. Feature Selection

Welche Sensoren wollen wir verwenden?

In [ ]:
# KONFIGURATION: Welche Sensoren pro Board nutzen?
# Für den Anfang: nur Sensor 0 pro Board (wie im LSTM-Training)
ACTIVE_SENSOR_IDS = [0]

print(f"Konfiguration: Nutze Sensoren {ACTIVE_SENSOR_IDS} pro Board.")

# 1. Ziel-Spalten (Winkel)
theta_cols = ["theta_1", "theta_2", "theta_3", "theta_4"]

# 2. Input-Features filtern
feature_cols = []

for c in df_total.columns:
    # Ignoriere Zeit und Winkel
    if c in ["t_ns"] + theta_cols:
        continue
        
    # Extrahiere Sensor ID aus dem Namen (z.B. 'x_b0_s0' -> 's0')
    parts = c.split('_')
    s_tag = parts[-1]  # Der letzte Teil ist immer s0, s1 etc.
    
    if s_tag.startswith('s'):
        try:
            s_id = int(s_tag.replace('s', ''))
            # Wenn die ID in unserer Wunschliste ist, behalten wir die Spalte
            if s_id in ACTIVE_SENSOR_IDS:
                feature_cols.append(c)
        except ValueError:
            pass  # Falls Spaltenname komisch ist

# 3. X und y erstellen
X = df_total[feature_cols].values
y = df_total[theta_cols].values

print("-" * 30)
print(f"Ausgewählte Features: {len(feature_cols)}")
print(f"Input Shape (X): {X.shape}")
print(f"Output Shape (y): {y.shape}")
print("-" * 30)
print("Beispiel Features:", feature_cols[:5])
print("\nDatensatz bereit für Polynomial Regression!")

## 4. Train/Test Split

In [ ]:
# 80/20 Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training Set: {X_train.shape[0]} Samples")
print(f"Test Set: {X_test.shape[0]} Samples")

## 5. Polynomial Features & Training

Wir testen verschiedene Polynomgrade (2, 3, evtl. 4)

In [ ]:
# Dictionary zum Speichern der Ergebnisse
results = {}

# Verschiedene Polynomgrade testen
degrees = [2, 3]  # Starte mit 2 und 3, 4 kann später getestet werden

for degree in degrees:
    print(f"\n{'='*50}")
    print(f"Training Polynomial Degree: {degree}")
    print(f"{'='*50}")
    
    # 1. Polynomial Features erstellen
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    
    # 2. Features transformieren
    print("Erstelle Polynomial Features...")
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    print(f"  Original Features: {X_train.shape[1]}")
    print(f"  Polynomial Features: {X_train_poly.shape[1]}")
    
    # 3. Features skalieren (wichtig für Ridge Regression)
    scaler = StandardScaler()
    X_train_poly_scaled = scaler.fit_transform(X_train_poly)
    X_test_poly_scaled = scaler.transform(X_test_poly)
    
    # 4. Modell trainieren (Ridge Regression gegen Overfitting)
    print("Trainiere Ridge Regression...")
    model = Ridge(alpha=1.0)  # Alpha kann optimiert werden
    
    start_time = time.time()
    model.fit(X_train_poly_scaled, y_train)
    train_time = time.time() - start_time
    
    print(f"  Training abgeschlossen in {train_time:.2f} Sekunden")
    
    # 5. Predictions
    start_time = time.time()
    y_pred_train = model.predict(X_train_poly_scaled)
    y_pred_test = model.predict(X_test_poly_scaled)
    inference_time = (time.time() - start_time) / len(X_test)  # Zeit pro Sample
    
    # 6. Evaluation
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_r2 = r2_score(y_test, y_pred_test)
    
    # 7. Ergebnisse speichern
    results[degree] = {
        'model': model,
        'poly': poly,
        'scaler': scaler,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'test_mae': test_mae,
        'test_r2': test_r2,
        'train_time': train_time,
        'inference_time_per_sample': inference_time,
        'n_features': X_train_poly.shape[1],
        'n_coefficients': model.coef_.size
    }
    
    # 8. Ausgabe
    print(f"\nErgebnisse für Degree {degree}:")
    print(f"  Train MSE: {train_mse:.6f}")
    print(f"  Test MSE:  {test_mse:.6f}")
    print(f"  Test MAE:  {test_mae:.6f}")
    print(f"  Test R²:   {test_r2:.6f}")
    print(f"  Anzahl Features: {X_train_poly.shape[1]}")
    print(f"  Anzahl Koeffizienten: {model.coef_.size}")
    print(f"  Inferenzzeit pro Sample: {inference_time*1000:.4f} ms")

print("\n" + "="*50)
print("Training abgeschlossen!")
print("="*50)

## 6. Vergleich der Modelle

In [ ]:
# Vergleichstabelle erstellen
comparison_df = pd.DataFrame([
    {
        'Degree': deg,
        'Test MSE': res['test_mse'],
        'Test MAE': res['test_mae'],
        'Test R²': res['test_r2'],
        'Features': res['n_features'],
        'Koeffizienten': res['n_coefficients'],
        'Inferenz (ms)': res['inference_time_per_sample'] * 1000,
        'Train Zeit (s)': res['train_time']
    }
    for deg, res in results.items()
])

print("\nVergleich der Polynomial Regression Modelle:")
print(comparison_df.to_string(index=False))

# Bestes Modell identifizieren
best_degree = comparison_df.loc[comparison_df['Test MSE'].idxmin(), 'Degree']
print(f"\nBestes Modell: Degree {int(best_degree)} (niedrigster Test MSE)")

## 7. Visualisierung der Predictions

In [ ]:
# Für das beste Modell Predictions visualisieren
best_model_data = results[int(best_degree)]
model = best_model_data['model']
poly = best_model_data['poly']
scaler = best_model_data['scaler']

# Predictions
X_test_poly = poly.transform(X_test)
X_test_poly_scaled = scaler.transform(X_test_poly)
y_pred = model.predict(X_test_poly_scaled)

# Plot für jeden Winkel
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for i, theta in enumerate(theta_cols):
    ax = axes[i]
    
    # Erste 500 Samples plotten für bessere Sichtbarkeit
    n_samples = min(500, len(y_test))
    
    ax.plot(y_test[:n_samples, i], label='Ground Truth', alpha=0.7)
    ax.plot(y_pred[:n_samples, i], label='Prediction', alpha=0.7)
    ax.set_xlabel('Sample')
    ax.set_ylabel('Winkel (Grad)')
    ax.set_title(f'{theta} - Polynomial Degree {int(best_degree)}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Fehlerverteilung
errors = y_test - y_pred

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for i, theta in enumerate(theta_cols):
    ax = axes[i]
    ax.hist(errors[:, i], bins=50, alpha=0.7, edgecolor='black')
    ax.set_xlabel('Fehler (Grad)')
    ax.set_ylabel('Häufigkeit')
    ax.set_title(f'{theta} - Fehlerverteilung')
    ax.axvline(0, color='red', linestyle='--', linewidth=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Modell speichern

In [ ]:
# Bestes Modell speichern
model_filename = f'polynomial_model_degree_{int(best_degree)}.pkl'
poly_filename = f'polynomial_features_degree_{int(best_degree)}.pkl'
scaler_filename = f'scaler_degree_{int(best_degree)}.pkl'

joblib.dump(best_model_data['model'], model_filename)
joblib.dump(best_model_data['poly'], poly_filename)
joblib.dump(best_model_data['scaler'], scaler_filename)

print(f"Modell gespeichert:")
print(f"  - {model_filename}")
print(f"  - {poly_filename}")
print(f"  - {scaler_filename}")

## 9. Zusammenfassung & Vergleich mit LSTM

### Polynomial Regression:
- ✅ **Sehr schnelle Inferenz** (Mikrosekunden)
- ✅ **Geringer Speicherbedarf** (nur Koeffizienten)
- ✅ **Deterministisch**
- ✅ **Einfache Deployment** (nur Matrixmultiplikation)
- ⚠️ **Feature Engineering nötig**
- ⚠️ **Overfitting-Risiko bei hohen Graden**

### LSTM:
- ✅ **Flexibler bei komplexen Mustern**
- ✅ **Kann zeitliche Abhängigkeiten lernen**
- ⚠️ **Langsamere Inferenz**
- ⚠️ **Größerer Speicherbedarf**
- ⚠️ **Schwerer zu deployen**

### Empfehlung:
Wenn die Polynomial Regression ähnliche Genauigkeit wie das LSTM erreicht, ist sie deutlich vorteilhafter für:
- Embedded Systems (z.B. Mikrocontroller)
- Echtzeitanwendungen
- Einfachere Wartung und Debugging